# boutRight Code Description

The boutRight project processes bird song recordings to detect and classify bouts using a YOLOv5 model. The code is divided into two main parts:

## Part 1: Setup and Function Definitions

1. **Import Libraries**: The script imports necessary libraries including `os`, `glob`, `shutil`, `torch`, `PIL`, `numpy`, `scipy`, `tqdm`, `io`, `IPython.display`, `concurrent.futures`, `uuid`, `pandas`, and `math`.

2. **Load YOLOv5 Model**: The YOLOv5 model is loaded with custom weights for detecting bouts in spectrogram images.

3. **Define Functions**:
    - `filtered_spectrogram`: Filters and generates a spectrogram from a given audio file. It applies a Butterworth filter, normalizes the signal, and calculates the spectrogram.
    - `check_for_bouts`: Generates a spectrogram image, uses the YOLOv5 model to detect bouts, and returns the detection results.
    - `process_wav_file`: Processes each WAV file, moves it to the appropriate directory (`Songs` or `Noise_Calls`), and appends the detection results to CSV files.
    - `append_to_csv`: Appends detection results to a CSV file, ensuring consistent data formatting.
    - `get_all_wav_files`: Retrieves all WAV files in a directory, including subdirectories.
    - `is_already_scanned`: Checks if a file has already been scanned by looking it up in a list of scanned files.

## Part 2: Processing Bird Song Recordings

1. **Set Base Directory**: The base directory containing bird song recordings is specified.

2. **Process Each Bird's Mic Folder**:
    - The script iterates through each bird's mic folder.
    - For each hatch day folder, it checks for the existence of `Songs` and `Noise_Calls` directories and creates them if they don't exist.
    - It loads the list of already scanned files from `wav_scanned.csv` if available.

3. **Process WAV Files**:
    - The script retrieves all WAV files in the hatch day folder, including subdirectories.
    - It uses a thread pool to process each WAV file concurrently.
    - For each file, it generates a spectrogram, detects bouts using the YOLOv5 model, and categorizes the files into `Songs` or `Noise_Calls`.
    - Detection results are appended to `results_bouts.csv` and `results_calls.csv`.

4. **Handle Duplicates**:
    - If a file has already been scanned and is detected as a duplicate, a warning is displayed, and the file is deleted.

5. **Save Scanned Files**:
    - The list of scanned files is saved to `wav_scanned.csv` to keep track of processed files and avoid reprocessing.

## Summary

This script is essential for preprocessing and analyzing bird song recordings, enabling the detection and classification of bouts. It ensures efficient processing, categorization, and management of bird song data, facilitating further analysis and research.


In [1]:
import os
from glob import glob
import shutil
import torch
from PIL import Image
import numpy as np
from scipy.io import wavfile
from scipy import signal
from tqdm import tqdm
import io
from IPython.display import clear_output
import concurrent.futures
import uuid
import pandas as pd
import math
import zipfile
import warnings
import threading
import hashlib

# Initialize a lock object
lock = threading.Lock()

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'custom', path=os.path.join(os.getcwd(),r'yolov5\runs\train\exp13\weights\best.pt'))
#folder to temporarily save images for YOLO detection
#this is needed to run parallel computing without problems
temp_path = r'C:\Temp'

# Function to filter and generate spectrogram
def filtered_spectrogram(filepath):
    # Length of FFT
    lend = 34
    # Overlap of FFT
    overlap = 33
    # Time length for exponential window of FFT
    ts = 3
    # Low cut frequency in Hz
    lc = 500
    # High cut frequency in Hz
    hc = 20000
    # Color of image settings
    # Contribution of each channel to color
    RGBch = [0.8, 1.5, 1.5]

    # Import the audio data
    fs, data = wavfile.read(filepath)
    # Round length of data and overlap
    lend = round((lend / 1E3) * fs)
    overlap = round((overlap / 1E3) * fs)
    # Next power of two definition
    def nextpow2(x):
        return 1 if x == 0 else 2**math.ceil(math.log2(x))
    # Calculate next power of two
    nfft = nextpow2(lend)

    # Butterworth filter
    def butter_bp(data, lc, hc, fs, order=3):
        nyq = 0.5 * fs
        low = lc / nyq
        high = hc / nyq
        b, a = signal.butter(order, [low, high], btype='band')
        data_filtered = signal.lfilter(b, a, data)
        return data_filtered

    data = butter_bp(data, lc, hc, fs, order=5)
    # Normalize signal
    data = data / max(abs(data))
    # Make windows for spectrogram
    t = np.linspace(-lend / 2 + 1, lend / 2, num=lend)
    sigma = (ts / 1E3) * fs
    w = np.exp(-(t / sigma)**2)
    dw = np.exp(-(t / (2 * sigma))**2)
    # Calculate spectrograms
    [f, t, sx] = signal.spectrogram(data, fs=fs, window=w, noverlap=overlap, nfft=nfft)
    [_, _, sxx] = signal.spectrogram(data, fs=fs, window=dw, noverlap=overlap, nfft=nfft)
    # Average of both spectrograms
    image_array = np.log2(abs(sx) + abs(sxx)) / 2
    # Obtain thresholds for background
    minmax = [np.percentile(image_array, 80), np.percentile(image_array, 99)]
    # Subtract background
    image_array = np.minimum(image_array, minmax[1])
    image_array = np.maximum(image_array, minmax[0])
    # Normalize
    image_array = (image_array - np.min(image_array)) / (np.max(image_array) - np.min(image_array))
    # Flip spectrogram
    image_array = np.flip(image_array, 0)
    # Convert to color
    sz = (image_array.shape[0] - 1, image_array.shape[1] - 1, 3)
    image_color = np.zeros(sz)
    tmp = image_array
    image_color[:, :, 0] = RGBch[0] * tmp[0:-1, 0:-1]
    tmp = np.diff(image_array, 1, axis=0)
    image_color[:, :, 1] = RGBch[1] * tmp[:, 0:-1]
    tmp = np.diff(image_array, 1, axis=1)
    image_color[:, :, 2] = RGBch[2] * tmp[0:-1, :]
    
    return image_color, fs, len(data)

# Function to generate spectrogram and check for bouts using YOLOv5
def check_for_bouts(wav_file, temp_path):
    temp_img_path = None
    try:
        # Ensure the temp_path directory exists
        os.makedirs(temp_path, exist_ok=True)
        
        # Generate the spectrogram and convert it to an image
        spectrogram, fs, data_length = filtered_spectrogram(wav_file)
        spectrogram_image = (spectrogram * 255).astype(np.uint8)
        pil_image = Image.fromarray(spectrogram_image)
        
        # Generate a unique filename for the temporary spectrogram image in the temp_path directory
        temp_img_filename = f'temp_spectrogram_{uuid.uuid4().hex}.png'
        temp_img_path = os.path.join(temp_path, temp_img_filename)
        pil_image.save(temp_img_path)
        
        # Verify the saved image
        with Image.open(temp_img_path) as img:
            img.verify()
        
        # Use YOLOv5 to detect bouts in the spectrogram image
        results = model(temp_img_path)
        
        # Filter detections to only include bouts (class 0)
        bouts = [bbox for bbox in results.xyxy[0] if bbox[5] == 0]
        
        # Check if any bouts are detected
        return len(bouts) > 0, results.xyxy[0], spectrogram.shape[1], data_length, fs
    except ValueError as e:
        if "File format" in str(e):
            print(f"Error in check_for_bouts for {wav_file}: {e}")
        else:
            raise e  # Re-raise other types of ValueErrors
        return False, [], 0, 0, 0
    except Exception as e:
        print(f"Error in check_for_bouts for {wav_file}: {e}")
        return False, [], 0, 0, 0
    finally:
        # Remove the temporary image file
        if temp_img_path and os.path.exists(temp_img_path):
            os.remove(temp_img_path)


# Function to process WAV file
def process_wav_file(wav_file, songs_dir, noise_calls_dir, bouts_csv_path, calls_csv_path, scanned_files, scanned_csv_path, processed_files):
    try:
        has_bouts, bboxes, spectrogram_length, data_length, fs = check_for_bouts(wav_file, temp_path)

        results = []
        for bbox in bboxes:
            x1, y1, x2, y2, conf, cls = bbox
            entry = {
                'wav_folder_path': os.path.dirname(wav_file),
                'wav_filename': os.path.basename(wav_file),
                'spectrogram_start_time': x1.item(),
                'spectrogram_end_time': x2.item(),
                'wav_file_start_time': (x1.item() / spectrogram_length) * (data_length / fs),
                'wav_file_end_time': (x2.item() / spectrogram_length) * (data_length / fs),
                'confidence': conf.item(),
                'label_class': cls.item(),
                'bbox_x1': x1.item(),
                'bbox_y1': y1.item(),
                'bbox_x2': x2.item(),
                'bbox_y2': y2.item()
            }
            results.append(entry)

        # Append results to CSV files
        if results:
            bouts_results = [entry for entry in results if entry['label_class'] == 0]
            calls_results = [entry for entry in results if entry['label_class'] == 1]

            with lock:
                if bouts_results:
                    append_to_csv(bouts_results, bouts_csv_path)
                if calls_results:
                    append_to_csv(calls_results, calls_csv_path)

        # Move the file based on detection results
        if has_bouts:
            shutil.move(wav_file, os.path.join(songs_dir, os.path.basename(wav_file)))
            with lock:
                processed_files.add(os.path.basename(wav_file))  # Thread-safe addition
            print(f"Moved {wav_file} to Songs")
        else:
            shutil.move(wav_file, os.path.join(noise_calls_dir, os.path.basename(wav_file)))
            print(f"Moved {wav_file} to Noise_Calls")

        # Update scanned files and save to CSV immediately
        with lock:
            scanned_files.append(os.path.basename(wav_file))
            pd.DataFrame({'filename': scanned_files}).to_csv(scanned_csv_path, index=False)

        return {
            'filepath': wav_file,
            'filename': os.path.basename(wav_file),
            'bboxes': bboxes,
            'spectrogram_length': spectrogram_length,
            'data_length': data_length,
            'fs': fs
        }
    except Exception as e:
        print(f"Error processing {wav_file}: {e}")
        return None

# Function to re-process WAV file that was missed in the original scan
def reprocess_wav_file(wav_file, noise_calls_dir, bouts_csv_path, calls_csv_path):
    try:
        has_bouts, bboxes, spectrogram_length, data_length, fs = check_for_bouts(wav_file, temp_path)

        results = []
        for bbox in bboxes:
            x1, y1, x2, y2, conf, cls = bbox
            entry = {
                'wav_folder_path': os.path.dirname(wav_file),
                'wav_filename': os.path.basename(wav_file),
                'spectrogram_start_time': x1.item(),
                'spectrogram_end_time': x2.item(),
                'wav_file_start_time': (x1.item() / spectrogram_length) * (data_length / fs),
                'wav_file_end_time': (x2.item() / spectrogram_length) * (data_length / fs),
                'confidence': conf.item(),
                'label_class': cls.item(),
                'bbox_x1': x1.item(),
                'bbox_y1': y1.item(),
                'bbox_x2': x2.item(),
                'bbox_y2': y2.item()
            }
            results.append(entry)

        # Append results to CSV files
        if results:
            bouts_results = [entry for entry in results if entry['label_class'] == 0]
            calls_results = [entry for entry in results if entry['label_class'] == 1]

            with lock:
                if bouts_results:
                    append_to_csv(bouts_results, bouts_csv_path)
                if calls_results:
                    append_to_csv(calls_results, calls_csv_path)

        if has_bouts:
            print(f"Reprocessed and found bouts in {wav_file}")
        else:
            print(f"Reprocessed {wav_file}, no bouts found")
            
            # Add this block to move files without bouts to the Noise_Calls folder
            shutil.move(wav_file, os.path.join(noise_calls_dir, os.path.basename(wav_file)))
            print(f"Moved {wav_file} to Noise_Calls after re-scan")

    except Exception as e:
        print(f"Error reprocessing {wav_file}: {e}")


# Function to append results to CSV
def append_to_csv(results, csv_path):
    # Define the expected columns
    columns = [
        'wav_folder_path', 'wav_filename', 'spectrogram_start_time', 'spectrogram_end_time',
        'wav_file_start_time', 'wav_file_end_time', 'confidence', 'label_class',
        'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'
    ]
    
    # Convert results to DataFrame
    new_df = pd.DataFrame(results, columns=columns)
    
    # Append to the CSV file
    if os.path.exists(csv_path):
        new_df.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        new_df.to_csv(csv_path, mode='w', header=True, index=False)

# Function to hash individual files to avoid duplicates
def hash_file(filepath):
    """Returns the MD5 hash of the file content."""
    hasher = hashlib.md5()
    with open(filepath, 'rb') as file:
        buf = file.read()
        hasher.update(buf)
    return hasher.hexdigest()

# Function to gather filename of wav files in folders and zip files
def get_all_wav_files(hatch_path, bird_path):
    wav_files = {}  # Use a dictionary to avoid duplicates by file content hash
    songs_dir = os.path.join(hatch_path, 'Songs')
    noise_calls_dir = os.path.join(hatch_path, 'Noise_Calls')

    # Extract ZIP files recursively
    zip_files = glob(os.path.join(hatch_path, '**', '*.zip'), recursive=True)
    for zip_file in zip_files:
        try:
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                for file_info in zip_ref.infolist():
                    # Check if it's a .wav file
                    if file_info.filename.endswith('.wav'):
                        try:
                            # Test individual file for corruption
                            zip_ref.extract(file_info, hatch_path)
                        except Exception as e:
                            print(f"Skipping corrupted file {file_info.filename} in {zip_file}: {e}")
                            continue  # Skip corrupted file and continue with the next one
        except zipfile.BadZipFile:
            print(f"Skipping corrupted ZIP file: {zip_file}")
            continue  # Skip the corrupted ZIP and move on

        # Move the ZIP file to the birdname folder (at the same level as the mic folder)
        shutil.move(zip_file, os.path.join(bird_path, os.path.basename(zip_file)))

    # Function to add files based on their content hash
    def add_wav_files(dir_path):
        for wav_file in glob(os.path.join(dir_path, '**', '*.wav'), recursive=True):
            file_hash = hash_file(wav_file)
            if file_hash not in wav_files:
                wav_files[file_hash] = wav_file
    
    # Add WAV files from the main directory, Songs, and Noise_Calls
    add_wav_files(hatch_path)
    if os.path.exists(songs_dir):
        add_wav_files(songs_dir)
    if os.path.exists(noise_calls_dir):
        add_wav_files(noise_calls_dir)
    
    return list(wav_files.values())  # Return only the unique file paths

# Function to check if a file has already been scanned
def is_already_scanned(wav_file, scanned_files):
    return os.path.basename(wav_file) in scanned_files

# Function to check if a song file is in the bouts CSV
def is_in_bouts_csv(wav_file, bouts_csv_path):
    if os.path.exists(bouts_csv_path):
        try:
            bouts_df = pd.read_csv(bouts_csv_path)
            return os.path.basename(wav_file) in bouts_df['wav_filename'].values
        except pd.errors.EmptyDataError:
            return False
    return False


Using cache found in C:\Users\ucsfg/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-4-30 Python-3.12.5 torch-2.4.0 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)

Fusing layers... 
Model summary: 166 layers, 7056607 parameters, 0 gradients
Adding AutoShape... 


In [ ]:
# Manually set the selected birds for scanning
# You can list one bird or multiple birds here
# Example: selected_birds = ['bird01', 'bird03'] or selected_birds = ['bird01'] for one bird
# If you want to process all birds, you can set selected_birds = ['all']

# Base directory and suppression of warnings remain unchanged
base_dir = r'Z:\Birdsong_560M'
warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")

# List all available birds
available_birds = [bird for bird in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, bird))]

# Manually set the birds you want to process
# Example: selected_birds = ['bird01', 'bird02']
selected_birds = available_birds  # Set to process all birds, or you can choose specific birds

# Main processing function for each bird
def process_bird(bird_name, base_dir):
    bird_path = os.path.join(base_dir, bird_name)
    mic_path = os.path.join(bird_path, 'mic')

    if not os.path.exists(mic_path):
        print(f"No 'mic' folder found for {bird_name}.")
        return

    for hatch_folder in os.listdir(mic_path):
        hatch_path = os.path.join(mic_path, hatch_folder)
        bird_name = bird_name  # Corrected this to use bird_name from function
        hatch_day = hatch_folder    

        if not os.path.isdir(hatch_path):
            continue

        songs_dir = os.path.join(hatch_path, 'Songs')
        noise_calls_dir = os.path.join(hatch_path, 'Noise_Calls')

        # Create Songs and Noise_Calls directories if they don't exist
        os.makedirs(songs_dir, exist_ok=True)
        os.makedirs(noise_calls_dir, exist_ok=True)

        bouts_csv_path = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_day}.csv')
        calls_csv_path = os.path.join(hatch_path, f'results_calls_{bird_name}_{hatch_day}.csv')
        scanned_csv_path = os.path.join(hatch_path, f'wav_scanned_{bird_name}_{hatch_day}.csv')
        print(f"Processing: {bouts_csv_path}")
        
        scanned_files = []
        if os.path.exists(scanned_csv_path):
            try:
                scanned_df = pd.read_csv(scanned_csv_path, on_bad_lines='skip')
                scanned_files = scanned_df['filename'].tolist()
            except pd.errors.EmptyDataError:
                scanned_files = []

        wav_files = get_all_wav_files(hatch_path, bird_path)

        # Filter out already scanned files
        unscanned_wav_files = [wav_file for wav_file in wav_files if not is_already_scanned(wav_file, scanned_files)]

        processed_files = set()  # Thread-safe modification

        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = {executor.submit(process_wav_file, wav_file, songs_dir, noise_calls_dir, bouts_csv_path, calls_csv_path, scanned_files, scanned_csv_path, processed_files): wav_file for wav_file in unscanned_wav_files}
            for i, future in enumerate(tqdm(concurrent.futures.as_completed(futures), total=len(unscanned_wav_files), desc="Processing WAV files")):
                if i % 10 == 0:
                    clear_output(wait=True)  # Clear the output every 10 files
                wav_file = futures[future]
                try:
                    result = future.result()
                except Exception as exc:
                    print(f'{wav_file} generated an exception: {exc}')

        # Final check: Ensure all files in Songs appear in both wav_scanned.csv and results_bouts.csv
        song_files_to_rescan = []
        for song_file in glob(os.path.join(songs_dir, '*.wav')):
            if not is_already_scanned(song_file, scanned_files):
                print(f"Appending {song_file} to scanned files")
                with lock:
                    scanned_files.append(os.path.basename(song_file))
                    pd.DataFrame({'filename': scanned_files}).to_csv(scanned_csv_path, index=False)
            if not is_in_bouts_csv(song_file, bouts_csv_path):
                print(f"File {song_file} missing from results_bouts.csv, adding to re-scan list")
                song_files_to_rescan.append(song_file)

        # Parallel processing for the re-scan of song files
        with concurrent.futures.ThreadPoolExecutor() as reprocess_executor:
            reprocess_futures = {reprocess_executor.submit(reprocess_wav_file, song_file, noise_calls_dir, bouts_csv_path, calls_csv_path): song_file for song_file in song_files_to_rescan}
            for i, future in enumerate(tqdm(concurrent.futures.as_completed(reprocess_futures), total=len(song_files_to_rescan), desc="Re-scanning Songs")):
                if i % 10 == 0:
                    clear_output(wait=True)  # Clear the output every 10 iterations
                song_file = reprocess_futures[future]
                try:
                    future.result()
                except Exception as exc:
                    print(f'{song_file} generated an exception during re-scan: {exc}')


# Process the selected birds
if selected_birds:
    for bird in selected_birds:
        # a=["EVC2303","8UWL2","B17Y4","B028","B033","B045","BK6","BK8W98","Bk19","Bk52W16"]
        # Found = [item for item in a if item == bird]
        # if Found:
        #     continue
        process_bird(bird, base_dir)
else:
    print("No birds selected. Exiting...")


Processing: Z:\Birdsong_560M\Bk18\mic\422\results_bouts_Bk18_422.csv


Processing WAV files: 0it [00:00, ?it/s]
Re-scanning Songs: 0it [00:00, ?it/s]


Processing: Z:\Birdsong_560M\Bk18\mic\423\results_bouts_Bk18_423.csv


Processing WAV files: 0it [00:00, ?it/s]
Re-scanning Songs: 0it [00:00, ?it/s]


Processing: Z:\Birdsong_560M\Bk18\mic\424\results_bouts_Bk18_424.csv


Processing WAV files: 0it [00:00, ?it/s]
Re-scanning Songs: 0it [00:00, ?it/s]


# Remove noise and call wav files which have the least amount of calls

This script processes bird song recordings to manage and retain the top 100 WAV files with the most calls in each hatch day folder. The steps are as follows:

1. **Import Libraries**: The script imports necessary libraries including `os`, `pandas`, and `glob`.

2. **Set Base Directory**: The base directory containing bird song recordings is specified.

3. **Define Function**:
    - `process_hatch_day_folder`: This function processes each hatch day folder by:
        - Checking if the `results_calls.csv` file exists.
        - Loading the CSV file and counting the number of calls in each WAV file.
        - Sorting the files by call count and retaining the top 100.
        - Deleting the remaining files in the `Noise_Calls` folder.

4. **Process Each Bird's Mic Folder**:
    - The script iterates through each bird's mic folder.
    - For each hatch day folder, it calls the `process_hatch_day_folder` function to manage the WAV files based on the number of calls.

This script helps in organizing and retaining the most relevant bird song recordings for further analysis.


In [2]:
import os
import csv
import pandas as pd
from glob import glob

# Base directory
base_dir = r'Z:\Birdsong_560M'

# Function to process each hatch day folder
def process_hatch_day_folder(hatch_path, bird_name, hatch_day):
    calls_csv_path = os.path.join(hatch_path, f'results_calls_{bird_name}_{hatch_day}.csv')
    noise_calls_dir = os.path.join(hatch_path, 'Noise_Calls')
    
    # Check if the calls CSV file exists
    if not os.path.exists(calls_csv_path):
        print(f"No CSV file found for {hatch_path}. Skipping...")
        return
    
    try:
        # Load the calls CSV file
        calls_df = pd.read_csv(calls_csv_path, on_bad_lines='skip')
        
        if calls_df.empty:
            print(f"No data in {calls_csv_path}. Skipping...")
            return
        
        # Group by filename and count the number of calls
        call_counts = calls_df.groupby('wav_filename').size().reset_index(name='call_count')
        
        # Sort by call count in descending order and keep the top 200
        top_calls = call_counts.sort_values(by='call_count', ascending=False).head(200)
        
        # Get the list of top 200 filenames
        top_filenames = top_calls['wav_filename'].tolist()
        
        # Get all WAV files in the Noise_Calls folder
        all_wav_files = glob(os.path.join(noise_calls_dir, '*.wav'))
        
        # Delete files not in the top 200
        for wav_file in all_wav_files:
            if os.path.basename(wav_file) not in top_filenames:
                os.remove(wav_file)
                print(f"Deleted {wav_file}")
    except pd.errors.EmptyDataError:
        print(f"EmptyDataError: No columns to parse from file {calls_csv_path}")

# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    for hatch_folder in os.listdir(mic_path):
        hatch_path = os.path.join(mic_path, hatch_folder)
        
        if not os.path.isdir(hatch_path):
            continue
        
        bird_name = bird_dir
        hatch_day = hatch_folder
        
        process_hatch_day_folder(hatch_path, bird_name, hatch_day)


def align_csv_columns(file_path, expected_columns=12):
    aligned_rows = []
    
    try:
        with open(file_path, 'r') as infile:
            reader = csv.reader(infile)
            for row in reader:
                # Count non-empty cells
                non_empty_cells = [cell for cell in row if cell]
                
                if len(non_empty_cells) == expected_columns:
                    # Shift the row to start at column 1 and ensure it has exactly 12 columns
                    aligned_row = non_empty_cells[:expected_columns]
                else:
                    aligned_row = row
                
                # Ensure the row has exactly 12 columns
                aligned_row = aligned_row[:expected_columns]
                
                aligned_rows.append(aligned_row)
        
        with open(file_path, 'w', newline='') as outfile:
            writer = csv.writer(outfile)
            writer.writerows(aligned_rows)
        print(f"CSV columns aligned successfully for {file_path}.")
    
    except Exception as e:
        print(f"An error occurred with {file_path}: {e}")

def process_hatch_folders(base_path, expected_columns=12):
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.startswith('results_bout') or file.startswith('results_call'):
                file_path = os.path.join(root, file)
                align_csv_columns(file_path, expected_columns)

# Fix excel files
process_hatch_folders(base_dir)


No CSV file found for Z:\Birdsong_560M\Bk18\mic\428. Skipping...
No CSV file found for Z:\Birdsong_560M\Bk18\mic\429. Skipping...
No CSV file found for Z:\Birdsong_560M\Bk18\mic\435. Skipping...
No CSV file found for Z:\Birdsong_560M\Bk18\mic\453. Skipping...
No CSV file found for Z:\Birdsong_560M\D079\mic\syntax_analysis. Skipping...
No CSV file found for Z:\Birdsong_560M\LB265\mic\images_for_training. Skipping...
No CSV file found for Z:\Birdsong_560M\LB286\mic\126. Skipping...
No CSV file found for Z:\Birdsong_560M\LB286\mic\127. Skipping...
No CSV file found for Z:\Birdsong_560M\R08\mic\387. Skipping...
No CSV file found for Z:\Birdsong_560M\R08\mic\421. Skipping...
No CSV file found for Z:\Birdsong_560M\R101\mic\350. Skipping...
No CSV file found for Z:\Birdsong_560M\R120\mic\390. Skipping...
Deleted Z:\Birdsong_560M\R13\mic\376\Noise_Calls\R13_45668.19052246_1_11_5_17_32.wav
Deleted Z:\Birdsong_560M\R13\mic\376\Noise_Calls\R13_45668.21727433_1_11_6_2_7.wav
Deleted Z:\Birdsong_560

## Bout-Triggered Spectrogram Image Generation
This section of the pipeline efficiently generates spectrogram images of vocal bouts for training data, preserving source information in the image filenames.
### Function: `generate_bout_segments`
Extracts time-indexed segments from WAV files based on annotated bout timing.
**Inputs:**
- `wav_file` — Path to the WAV file.
- `bout_entries` — List of dicts, each with `wav_file_start_time` and `wav_file_end_time`.
- `pre_buffer` and `post_buffer` — Seconds of context to include around the bout.
**Returns:**
- A list of tuples with WAV path, index ranges, sample rate, data type, and original bout timing.
### Function: `save_bout_images`
Converts audio segments into spectrogram images and names them based on source metadata.
**Behavior:**
- Writes each bout segment to a temporary WAV file.
- Generates a color spectrogram using `filtered_spectrogram()`.
- Saves as `.png` in the format:


In [ ]:
# NEW FUNCTION: Generate spectrogram segments for a single file
def generate_bout_segments(wav_file, bout_entries, pre_buffer=1.0, post_buffer=1.0):
    try:
        fs, data = wavfile.read(wav_file)
        segments = []
        for entry in bout_entries:
            start_sec = max(0.0, entry['wav_file_start_time'] - pre_buffer)
            end_sec = min(len(data) / fs, entry['wav_file_end_time'] + post_buffer)
            start_idx = int(start_sec * fs)
            end_idx = int(end_sec * fs)
            segments.append((wav_file, start_idx, end_idx, fs, data.dtype, entry['wav_file_start_time'], entry['wav_file_end_time']))
        return segments
    except Exception as e:
        print(f"Error collecting bout segments for {wav_file}: {e}")
        return []

# Function to save a list of segments as spectrogram images
def save_bout_images(segments, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i, (wav_file, start_idx, end_idx, fs, dtype, bout_start, bout_end) in enumerate(segments):
        try:
            fs, data = wavfile.read(wav_file)
            segment = data[start_idx:end_idx]
            temp_wav_path = os.path.join(out_dir, f"temp_segment_{i}.wav")
            wavfile.write(temp_wav_path, fs, segment.astype(dtype))
            spectrogram, _, _ = filtered_spectrogram(temp_wav_path)
            img = Image.fromarray((spectrogram * 255).astype(np.uint8))

            wav_base = os.path.splitext(os.path.basename(wav_file))[0]
            img_name = f"{wav_base}_from_{int(bout_start)}s_to_{int(bout_end)}s.png"
            img_path = os.path.join(out_dir, img_name)

            img.save(img_path)
            img.close()
            os.remove(temp_wav_path)
        except Exception as e:
            print(f"Failed to save image {i}: {e}")
    print(f"Saved {len(segments)} images to {out_dir}")

# Optimized version: pick random WAVs with bouts and generate up to 30 images
base_dir = r"Z:\Birdsong_560M"
selected_birds = ['LB220']

for bird in selected_birds:
    bird_path = os.path.join(base_dir, bird)
    mic_path = os.path.join(bird_path, 'mic')
    if not os.path.isdir(mic_path): continue

    bout_entries_by_wav = {}

    for hatch in os.listdir(mic_path):
        hatch_path = os.path.join(mic_path, hatch)
        if not os.path.isdir(hatch_path): continue

        bouts_csv = os.path.join(hatch_path, f'results_bouts_{bird}_{hatch}.csv')
        if not os.path.exists(bouts_csv): continue

        try:
            bouts_df = pd.read_csv(bouts_csv)
            for wav_name, group in bouts_df.groupby('wav_filename'):
                wav_path = os.path.join(hatch_path, 'Songs', wav_name)
                if os.path.exists(wav_path):
                    bout_entries_by_wav[wav_path] = group.to_dict('records')
        except Exception as e:
            print(f"Error reading {bouts_csv}: {e}")

    wav_files = list(bout_entries_by_wav.keys())
    np.random.shuffle(wav_files)

    output_img_dir = os.path.join(mic_path, 'images_for_training')
    saved_segments = []

    for wav_file in wav_files:
        if len(saved_segments) >= 30:
            break
        segments = generate_bout_segments(wav_file, bout_entries_by_wav[wav_file])
        if segments:
            np.random.shuffle(segments)
            saved_segments.append(segments[0])

    save_bout_images(saved_segments, output_img_dir)

# Simple bout analytics

This script processes bird song recordings to generate and save plots for the distribution of bout durations and YOLO confidence scores. The steps are as follows:

1. **Import Libraries**: The script imports necessary libraries including `pandas`, `matplotlib.pyplot`, and `os`.

2. **Define Functions**:
    - `plot_bout_durations`: This function calculates and plots the distribution of bout durations from the `results_bouts.csv` file. It saves the plot as an SVG file.
    - `plot_yolo_confidence`: This function calculates and plots the distribution of YOLO confidence scores from the `results_bouts.csv` and `results_calls.csv` files. It saves the plot as an SVG file.

3. **Set Base Directory**: The base directory containing bird song recordings is specified.

4. **Process Each Bird's Mic Folder**:
    - The script iterates through each bird's mic folder.
    - For each hatch day folder, it checks for the existence of `Songs` and `Noise_Calls` directories.
    - It loads the results from `results_bouts.csv` and `results_calls.csv` if available.
    - It calls the `plot_bout_durations` and `plot_yolo_confidence` functions to generate and save the plots.

5. **Output**: The script prints a message indicating that the plots have been generated and saved successfully.

This script helps in visualizing the distribution of bout durations and YOLO confidence scores for bird song recordings.


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os
base_dir = r'Z:\Birdsong_560M'

# Base directory
# Function to plot and save the distribution of bout durations
def plot_bout_durations(hatch_path, bird_name, hatch_number):
    bouts_file = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_number}.csv')
    if os.path.exists(bouts_file):
        try:
            bouts_df = pd.read_csv(bouts_file)
            if bouts_df.empty:
                print(f"No data in {bouts_file}")
                return
            bouts_df['duration'] = bouts_df['wav_file_end_time'] - bouts_df['wav_file_start_time']
            
            plt.figure(figsize=(10, 6))
            plt.hist(bouts_df['duration'], bins=30, edgecolor='black')
            plt.title(f'Bout Duration Distribution for {bird_name} Hatch {hatch_number}')
            plt.xlabel('Duration (seconds)')
            plt.ylabel('Frequency')
            plt.savefig(os.path.join(hatch_path, 'bout_duration_distribution.svg'))
            plt.close()
        except pd.errors.EmptyDataError:
            print(f"EmptyDataError: No columns to parse from file {bouts_file}")

# Function to plot and save the distribution of YOLO confidence
def plot_yolo_confidence(hatch_path, bird_name, hatch_number):
    bouts_file = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_number}.csv')
    calls_file = os.path.join(hatch_path, f'results_calls_{bird_name}_{hatch_number}.csv')
    
    bouts_confidences = []
    calls_confidences = []
    
    if os.path.exists(bouts_file):
        try:
            bouts_df = pd.read_csv(bouts_file, on_bad_lines='skip')
            if not bouts_df.empty and 'confidence' in bouts_df.columns:
                bouts_confidences.extend(bouts_df['confidence'].tolist())
        except pd.errors.EmptyDataError:
            print(f"EmptyDataError: No columns to parse from file {bouts_file}")
    
    if os.path.exists(calls_file):
        try:
            calls_df = pd.read_csv(calls_file, on_bad_lines='skip')
            if not calls_df.empty and 'confidence' in calls_df.columns:
                calls_confidences.extend(calls_df['confidence'].tolist())
        except pd.errors.EmptyDataError:
            print(f"EmptyDataError: No columns to parse from file {calls_file}")
    
    if bouts_confidences or calls_confidences:
        plt.figure(figsize=(10, 6))
        plt.hist(bouts_confidences, bins=30, alpha=0.5, color='blue', label='Bouts')
        plt.hist(calls_confidences, bins=30, alpha=0.5, color='red', label='Calls')
        plt.title(f'YOLO Confidence Distribution for {bird_name} Hatch {hatch_number}')
        plt.xlabel('Confidence')
        plt.ylabel('Frequency')
        plt.legend(loc='upper right')
        plt.savefig(os.path.join(hatch_path, 'bout_confidence_distribution.svg'))
        plt.close()


# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    for hatch_folder in os.listdir(mic_path):
        hatch_path = os.path.join(mic_path, hatch_folder)
        
        if not os.path.isdir(hatch_path):
            continue
        
        bird_name = bird_dir
        hatch_number = hatch_folder
        
        # Plot and save the distribution of bout durations
        plot_bout_durations(hatch_path, bird_name, hatch_number)
        
        # Plot and save the distribution of YOLO confidence
        plot_yolo_confidence(hatch_path, bird_name, hatch_number)

print("Plots have been generated and saved successfully.")


Plots have been generated and saved successfully.


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import numpy as np

# Base directory

# Function to plot and save the heatmap of bout duration distributions across hatch days
def plot_bout_duration_heatmap(bird_path, bird_name):
    mic_path = os.path.join(bird_path, 'mic')
    hatch_folders = [f for f in os.listdir(mic_path) if os.path.isdir(os.path.join(mic_path, f))]
    
    # Dictionary to store bout durations for each hatch day
    bout_durations = {}
    
    for hatch_folder in hatch_folders:
        hatch_path = os.path.join(mic_path, hatch_folder)
        bouts_file = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_folder}.csv')
        
        if os.path.exists(bouts_file):
            try:
                bouts_df = pd.read_csv(bouts_file)
                if not bouts_df.empty:
                    bouts_df['duration'] = bouts_df['wav_file_end_time'] - bouts_df['wav_file_start_time']
                    bout_durations[hatch_folder] = bouts_df['duration'].tolist()
            except pd.errors.EmptyDataError:
                print(f"EmptyDataError: No columns to parse from file {bouts_file}")
    
    if not bout_durations:
        print(f"No valid bout data found for {bird_name}")
        return
    
    # Create a DataFrame from the dictionary
    max_duration = 6  # Limit the duration to 6 seconds
    num_bins = 20
    duration_bins = pd.cut([0, max_duration], bins=num_bins, retbins=True)[1]  # Get the bin edges
    
    heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)
    
    for hatch_folder, durations in bout_durations.items():
        # Filter durations to be within 0 to 6 seconds
        filtered_durations = [d for d in durations if 0 <= d <= max_duration]
        if filtered_durations:
            duration_counts = pd.cut(filtered_durations, bins=duration_bins).value_counts()
            heatmap_data.loc[hatch_folder] = duration_counts.values
    
    # Normalize the data to represent probabilities
    heatmap_data = heatmap_data.div(heatmap_data.sum(axis=1), axis=0).fillna(0)
    
    # Plot the heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(heatmap_data, cmap="YlGnBu", cbar_kws={'label': 'Probability'}, xticklabels=[f'{x:.2f}' for x in duration_bins[:-1]])
    plt.title(f'Bout Duration Distribution Heatmap for {bird_name}')
    plt.xlabel('Duration (seconds)')
    plt.ylabel('Hatch Day')
    # Rotate x-axis labels vertically
    plt.xticks(rotation=90)
    # Save the heatmap as a vector image
    heatmap_file = os.path.join(bird_path, 'bout_duration_heatmap.svg')
    plt.savefig(heatmap_file)
    plt.close()
    print(f"Heatmap saved for {bird_name} at {heatmap_file}")


# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    bird_name = bird_dir
    
    # Plot and save the heatmap of bout duration distributions across hatch days
    plot_bout_duration_heatmap(bird_path, bird_name)

print("Heatmaps have been generated and saved successfully.")

# Function to plot and save the number of bouts detected per hatch day
def plot_bout_counts_per_hatch_day(bird_path, bird_name):
    mic_path = os.path.join(bird_path, 'mic')
    hatch_folders = [f for f in os.listdir(mic_path) if os.path.isdir(os.path.join(mic_path, f))]
    
    # Dictionary to store the number of bouts for each hatch day
    bout_counts = {}

    for hatch_folder in hatch_folders:
        hatch_path = os.path.join(mic_path, hatch_folder)
        bouts_file = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_folder}.csv')
        
        if os.path.exists(bouts_file):
            try:
                bouts_df = pd.read_csv(bouts_file)
                if not bouts_df.empty:
                    # Count the number of bouts for this hatch day
                    bout_counts[hatch_folder] = len(bouts_df)
            except pd.errors.EmptyDataError:
                print(f"EmptyDataError: No columns to parse from file {bouts_file}")
    
    if not bout_counts:
        print(f"No valid bout data found for {bird_name}")
        return
    
    # Sort bout counts by hatch day
    bout_counts = {k: v for k, v in sorted(bout_counts.items(), key=lambda item: int(item[0]))}
    
    # Plot the bout counts
    plt.figure(figsize=(10, 6))
    plt.bar(bout_counts.keys(), bout_counts.values(), color='skyblue', edgecolor='black')
    plt.title(f'Number of Bouts Detected per Hatch Day for {bird_name}')
    plt.xlabel('Hatch Day')
    plt.ylabel('Number of Bouts')
    # Rotate x-axis labels vertically
    plt.xticks(rotation=90)
    # Save the figure as an SVG file
    bouts_count_svg = os.path.join(bird_path, 'bout_counts_per_hatch_day.svg')
    plt.savefig(bouts_count_svg)
    plt.close()
    print(f"SVG plot saved for {bird_name} at {bouts_count_svg}")


# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    bird_name = bird_dir
    
    # Plot and save the number of bouts detected per hatch day
    plot_bout_counts_per_hatch_day(bird_path, bird_name)

print("Bout count plots have been generated and saved successfully.")

# Function to extract the hour from the filename
def extract_hour_from_filename(filename):
    # Assuming the filename format is B028_45423.68911763_5_11_19_8_31.wav
    match = re.search(r'_(\d+)_(\d+)_(\d+)_(\d+)_\d+\.wav', filename)  # Extracting the part with _5_11_19_8_31
    if match:
        hour = match.group(3)  # The third group (19 in the example) is the hour
        return int(hour)
    return None

# Function to calculate and plot the average number of bouts per hour of the day
def plot_avg_bouts_per_hour(bird_path, bird_name):
    mic_path = os.path.join(bird_path, 'mic')
    hatch_folders = [f for f in os.listdir(mic_path) if os.path.isdir(os.path.join(mic_path, f))]
    
    # Dictionary to store bout counts per hour for each hatch day
    bouts_per_hour = {hour: [] for hour in range(24)}  # A dictionary of lists to collect counts for each hatch day
    
    for hatch_folder in hatch_folders:
        hatch_path = os.path.join(mic_path, hatch_folder)
        bouts_file = os.path.join(hatch_path, f'results_bouts_{bird_name}_{hatch_folder}.csv')
        
        if os.path.exists(bouts_file):
            try:
                bouts_df = pd.read_csv(bouts_file)
                if not bouts_df.empty:
                    # Create a temporary dictionary to count the bouts for each hour in this hatch day
                    hatch_bouts_per_hour = {hour: 0 for hour in range(24)}
                    
                    # Extract the hour from the filename for each bout and count it
                    for _, row in bouts_df.iterrows():
                        hour = extract_hour_from_filename(row['wav_filename'])
                        if hour is not None:
                            hatch_bouts_per_hour[hour] += 1  # Increment the count for the detected hour
                    
                    # Append the counts for each hour to the main bouts_per_hour dictionary
                    for hour, count in hatch_bouts_per_hour.items():
                        bouts_per_hour[hour].append(count)  # Append the count for this hour
            except pd.errors.EmptyDataError:
                print(f"EmptyDataError: No columns to parse from file {bouts_file}")
    
    # Calculate the average and standard deviation of bouts per hour across all hatch days
    avg_bouts_per_hour = []
    std_bouts_per_hour = []
    
    for hour in range(24):
        if bouts_per_hour[hour]:
            avg_bouts_per_hour.append(np.mean(bouts_per_hour[hour]))
            std_bouts_per_hour.append(np.std(bouts_per_hour[hour]))
        else:
            avg_bouts_per_hour.append(0)
            std_bouts_per_hour.append(0)
    
    # Plot the average number of bouts per hour and the standard deviation as a shadow
    hours = range(24)
    avg_bouts_per_hour = np.array(avg_bouts_per_hour)
    std_bouts_per_hour = np.array(std_bouts_per_hour)
    
    plt.figure(figsize=(10, 6))
    
    # Plot the average line
    plt.plot(hours, avg_bouts_per_hour, 'b-', label='Average Bouts')
    
    # Plot the shaded area for standard deviation
    plt.fill_between(hours, avg_bouts_per_hour - std_bouts_per_hour, avg_bouts_per_hour + std_bouts_per_hour, color='blue', alpha=0.2, label='Standard Deviation')
    
    plt.title(f'Average Number of Bouts per Hour for {bird_name}')
    plt.xlabel('Hour of the Day')
    plt.ylabel('Average Number of Bouts')
    plt.xticks(hours)
    plt.grid(True)
    
    # Save the figure as an SVG file
    avg_bouts_svg = os.path.join(bird_path, 'avg_bouts_per_hour.svg')
    plt.savefig(avg_bouts_svg)
    plt.close()
    print(f"SVG plot saved for {bird_name} at {avg_bouts_svg}")

# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    bird_name = bird_dir
    
    # Plot and save the average number of bouts per hour of the day
    plot_avg_bouts_per_hour(bird_path, bird_name)

print("Average bouts per hour plots have been generated and saved successfully.")


C:\Users\ucsfg\AppData\Local\Temp\ipykernel_21632\2531698845.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)


Heatmap saved for Bk18 at Z:\Birdsong_560M\Bk18\bout_duration_heatmap.svg


C:\Users\ucsfg\AppData\Local\Temp\ipykernel_21632\2531698845.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)


Heatmap saved for D079 at Z:\Birdsong_560M\D079\bout_duration_heatmap.svg


C:\Users\ucsfg\AppData\Local\Temp\ipykernel_21632\2531698845.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)


...

Heatmap saved for R90 at Z:\Birdsong_560M\R90\bout_duration_heatmap.svg


C:\Users\ucsfg\AppData\Local\Temp\ipykernel_21632\2531698845.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)


Heatmap saved for R97 at Z:\Birdsong_560M\R97\bout_duration_heatmap.svg
No valid bout data found for W01


C:\Users\ucsfg\AppData\Local\Temp\ipykernel_21632\2531698845.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  heatmap_data = pd.DataFrame(index=hatch_folders, columns=range(num_bins)).fillna(0)


Heatmap saved for W36 at Z:\Birdsong_560M\W36\bout_duration_heatmap.svg
Heatmaps have been generated and saved successfully.
SVG plot saved for Bk18 at Z:\Birdsong_560M\Bk18\bout_counts_per_hatch_day.svg
SVG plot saved for D079 at Z:\Birdsong_560M\D079\bout_counts_per_hatch_day.svg
SVG plot saved for LB225 at Z:\Birdsong_560M\LB225\bout_counts_per_hatch_day.svg
SVG plot saved for LB236 at Z:\Birdsong_560M\LB236\bout_counts_per_hatch_day.svg
SVG plot saved for LB265 at Z:\Birdsong_560M\LB265\bout_counts_per_hatch_day.svg
SVG plot saved for LB286 at Z:\Birdsong_560M\LB286\bout_counts_per_hatch_day.svg
SVG plot saved for R08 at Z:\Birdsong_560M\R08\bout_counts_per_hatch_day.svg
SVG plot saved for R101 at Z:\Birdsong_560M\R101\bout_counts_per_hatch_day.svg
SVG plot saved for R104 at Z:\Birdsong_560M\R104\bout_counts_per_hatch_day.svg
SVG plot saved for R106 at Z:\Birdsong_560M\R106\bout_counts_per_hatch_day.svg
SVG plot saved for R120 at Z:\Birdsong_560M\R120\bout_counts_per_hatch_day.svg


# bout Entropy analytics

This script processes bird song recordings to calculate and visualize the spectral entropy of detected bouts. The steps are as follows:

1. **Import Libraries**: The script imports necessary libraries including `os`, `pandas`, `numpy`, `scipy`, and `matplotlib`.

2. **Define Function**:
    - `calculate_spectral_entropy`: This function computes the spectral entropy of a given signal using the Short-Time Fourier Transform (STFT).

3. **Set Base Directory**: The base directory containing bird song recordings is specified.

4. **Process Each Bird's Mic Folder**:
    - The script iterates through each bird's mic folder.
    - For each hatch day folder, it checks for the existence of `Songs` and `Noise_Calls` directories.
    - It loads the results from `results_bouts.csv` if available.

5. **Process Each Bout**:
    - For each bout in the CSV file, the corresponding WAV file is read.
    - A region of interest (ROI) around the bout is extracted with 1-second padding.
    - The spectral entropy of the bout signal is calculated and stored.

6. **Sort and Align Bouts**:
    - The bouts are sorted based on their total spectral entropy.
    - All bouts are aligned to a common signal at the start of the bout using cross-correlation.

7. **Visualize Spectral Entropy**:
    - A figure is created where each row represents a bout, the x-axis is time, and the color represents the spectral entropy.
    - The figure is saved as an SVG file and displayed.

This script helps in analyzing and visualizing the complexity of bird songs by calculating and aligning the spectral entropy of detected bouts.


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.io import wavfile
from scipy.signal import spectrogram
import matplotlib.pyplot as plt

# Function to calculate spectral entropy
def calculate_spectral_entropy(signal, fs, nperseg=256, noverlap=128):
    f, t, Sxx = spectrogram(signal, fs, nperseg=nperseg, noverlap=noverlap)
    Sxx_norm = Sxx / np.sum(Sxx, axis=0)
    spectral_entropy = -np.sum(Sxx_norm * np.log2(Sxx_norm + 1e-10), axis=0)
    return spectral_entropy

# Base directory
base_dir = r'D:\Lab Dropbox\BirdSong\BirdData_2024'

# Process each bird's mic folder
for bird_dir in os.listdir(base_dir):
    bird_path = os.path.join(base_dir, bird_dir)
    mic_path = os.path.join(bird_path, 'mic')
    
    if not os.path.exists(mic_path):
        continue
    
    for hatch_folder in os.listdir(mic_path):
        hatch_path = os.path.join(mic_path, hatch_folder)
        
        if not os.path.isdir(hatch_path):
            continue
        
        songs_dir = os.path.join(hatch_path, 'Songs')
        noise_calls_dir = os.path.join(hatch_path, 'Noise_Calls')
        
        if not os.path.exists(songs_dir) or not os.path.exists(noise_calls_dir):
            continue
        
        # Load results from CSV
        results_csv = os.path.join(hatch_path, 'results_bouts.csv')
        if not os.path.exists(results_csv):
            continue
        
        results_df = pd.read_csv(results_csv)

        # Process each bout
        bouts = []
        for index, row in results_df.iterrows():
            wav_file = row['wav_filename']
            wav_file_path = os.path.join(songs_dir, wav_file)
            try:
                fs, data = wavfile.read(wav_file_path)
            except PermissionError as e:
                print(f"PermissionError: {e}")
                continue
            
            roi_wav_start = int(row['wav_file_start_time'] * fs)
            roi_wav_end = int(row['wav_file_end_time'] * fs)
            
            # Add 1 second padding at start and end of each bout
            start_idx = max(0, roi_wav_start - fs)
            end_idx = min(len(data), roi_wav_end + fs)
            
            bout_signal = data[start_idx:end_idx]
            spectral_entropy = calculate_spectral_entropy(bout_signal, fs)
            
            bouts.append({
                'spectral_entropy': spectral_entropy,
                'total_spectral_entropy': np.sum(spectral_entropy),
                'start_idx': start_idx,
                'end_idx': end_idx,
                'wav_file': wav_file_path
            })

        # Sort bouts based on total summed spectral entropy
        bouts.sort(key=lambda x: x['total_spectral_entropy'], reverse=True)

        # Align all bouts to the common signal at the start of the bout
        aligned_bouts = []
        reference_bout = bouts[0]['spectral_entropy']
        for bout in bouts:
            correlation = np.correlate(reference_bout, bout['spectral_entropy'], mode='full')
            shift = np.argmax(correlation) - len(reference_bout) + 1
            aligned_bout = np.roll(bout['spectral_entropy'], shift)
            aligned_bouts.append(aligned_bout)

        # Make a figure in which each row is a bout, the x-axis is time, and the color is the spectral entropy
        plt.figure(figsize=(10, len(aligned_bouts)))
        for i, aligned_bout in enumerate(aligned_bouts):
            plt.imshow(aligned_bout[np.newaxis, :], aspect='auto', cmap='viridis', extent=[0, len(aligned_bout) / fs, i, i + 1])
        
        plt.xlabel('Time (s)')
        plt.ylabel('Bout')
        plt.yticks(np.arange(len(aligned_bouts)) + 0.5, np.arange(1, len(aligned_bouts) + 1))
        plt.gca().invert_yaxis()
        plt.tight_layout()
        output_svg = os.path.join(hatch_path, 'bouts_spectral_entropy.svg')
        plt.savefig(output_svg, format='svg')
        plt.show()
